# Exploring the Sample Dataset

This notebook explores cloud physical properties retrieved by **clouds-decoded** from a
sample dataset of Sentinel-2 L1C imagery spanning January 2016 – December 2025 across
four tiles:

| Tile | Location |
|------|----------|
| T02KNG | Samoa (South Pacific) |
| T11RLP | California (USA) |
| T21PTT | Caribbean |
| T54SVC | Japan |

Each scene was processed through the full pipeline: cloud mask → cloud height →
albedo → refocus → cloud properties.

We use the project's **DuckDB database** for tabular statistics and the
**self-describing GeoTIFF metadata** to understand what each output file contains.

The dataset only contains 1 image per month for each of the tiles, so results are
quite noisy. Nevertheless, some trends can still be seen that relate to general
yearly trends in these locations.

## 0. Download the dataset

The sample dataset is hosted on Hugging Face at
[asterisk-labs/clouds-decoded-samples](https://huggingface.co/datasets/asterisk-labs/clouds-decoded-samples/).
The cell below downloads it into a local `sample_dataset/` directory. This only
needs to run once — subsequent runs will skip already-downloaded files.

In [ ]:
from pathlib import Path

DATASET_REPO = "asterisk-labs/clouds-decoded-samples"
PROJECT_DIR = Path("sample_dataset")

if (PROJECT_DIR / "project.db").exists():
    print(f"Dataset already present at {PROJECT_DIR}/")
else:
    from huggingface_hub import snapshot_download

    print(f"Downloading dataset from huggingface.co/datasets/{DATASET_REPO} ...")
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        local_dir=str(PROJECT_DIR),
    )
    print(f"Done — downloaded to {PROJECT_DIR}/")

In [ ]:
from __future__ import annotations

import json
import re

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

In [ ]:
# --- Configuration ---
OUTPUT_DIR = PROJECT_DIR / "outputs"
DB_PATH = PROJECT_DIR / "project.db"

db = duckdb.connect(str(DB_PATH), read_only=True)

MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
TILE_LABELS = {"T02KNG": "Samoa", "T11RLP": "California",
               "T21PTT": "Caribbean", "T54SVC": "Japan"}

## 1. Dataset overview from the project database

The project database tracks every processing run: scene metadata, run status, and
per-step statistics. Let's query it to understand what we have.

In [ ]:
# What tiles do we have, and how many completed scenes per tile?
db.sql("""
    SELECT
        sm.tile_id,
        COUNT(*) AS n_scenes,
        MIN(sm.sensing_time)::DATE AS earliest,
        MAX(sm.sensing_time)::DATE AS latest,
        ROUND(AVG(sm.sun_zenith), 1) AS mean_sza
    FROM runs r
    JOIN scene_metadata sm ON r.scene_id = sm.scene_id
    WHERE r.status = 'done'
    GROUP BY sm.tile_id
    ORDER BY sm.tile_id
""").df()

In [ ]:
# Cloud cover summary per tile
db.sql("""
    SELECT
        sm.tile_id,
        ROUND(AVG(cm.cloud_frac) * 100, 1) AS mean_cloud_pct,
        ROUND(MIN(cm.cloud_frac) * 100, 1) AS min_cloud_pct,
        ROUND(MAX(cm.cloud_frac) * 100, 1) AS max_cloud_pct,
        ROUND(MEDIAN(cm.cloud_frac) * 100, 1) AS median_cloud_pct
    FROM stats_cloud_mask cm
    JOIN runs r ON cm.run_id = r.run_id
    JOIN scene_metadata sm ON r.scene_id = sm.scene_id
    GROUP BY sm.tile_id
    ORDER BY sm.tile_id
""").df()

In [ ]:
# Monthly cloud fraction from the database
monthly = db.sql("""
    SELECT
        sm.tile_id,
        EXTRACT(MONTH FROM sm.sensing_time::TIMESTAMP) AS month,
        AVG(cm.cloud_frac) AS mean_cloud_frac
    FROM stats_cloud_mask cm
    JOIN runs r ON cm.run_id = r.run_id
    JOIN scene_metadata sm ON r.scene_id = sm.scene_id
    GROUP BY sm.tile_id, month
    ORDER BY sm.tile_id, month
""").df()

fig, ax = plt.subplots(figsize=(11, 5))
for tile_id, group in monthly.groupby("tile_id"):
    ax.plot(group["month"], group["mean_cloud_frac"] * 100, "o-",
            label=f"{tile_id} ({TILE_LABELS.get(tile_id, '')})",
            linewidth=2, markersize=5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel("Cloud fraction (%)")
ax.set_title("Monthly mean cloud fraction (from cloud mask statistics)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 12.5)
fig.tight_layout()
plt.show()

In [ ]:
# Cloud properties time series: median optical depth and mean ice/liquid ratio
props_ts = db.sql("""
    SELECT
        sm.tile_id,
        sm.sensing_time::TIMESTAMP AS time,
        cp.tau__p050 AS tau_median,
        cp.ice_liq_ratio__mean AS ilr_mean
    FROM stats_cloud_properties cp
    JOIN runs r ON cp.run_id = r.run_id
    JOIN scene_metadata sm ON r.scene_id = sm.scene_id
    ORDER BY sm.tile_id, time
""").df()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ROLLING_WINDOW = 6  # 6-scene rolling average (~6 months)

for tile_id, group in props_ts.groupby("tile_id"):
    group = group.sort_values("time")
    label = f"{tile_id} ({TILE_LABELS.get(tile_id, '')})"

    tau_smooth = group["tau_median"].rolling(ROLLING_WINDOW, center=True, min_periods=2).mean()
    ilr_smooth = group["ilr_mean"].rolling(ROLLING_WINDOW, center=True, min_periods=2).mean()

    ax1.plot(group["time"], tau_smooth, linewidth=2, label=label)
    ax2.plot(group["time"], ilr_smooth, linewidth=2, label=label)

ax1.set_ylabel("Median optical depth (\u03c4)")
ax1.set_title(f"Cloud optical depth and ice/liquid ratio over time ({ROLLING_WINDOW}-scene rolling mean)")
ax1.legend(fontsize=8, ncols=2)
ax1.grid(True, alpha=0.3)

ax2.set_ylabel("Mean ice/liquid ratio")
ax2.set_xlabel("Date")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Understanding output files via embedded metadata

Every GeoTIFF written by clouds-decoded carries a `clouds_decoded` tag with full
provenance: which pipeline step produced it, the configuration used, and the
source scene. This means we never need to hardcode band indices — the metadata
tells us what each band contains.

In [ ]:
def read_provenance(path: Path) -> dict:
    """Read the clouds_decoded provenance tag from a GeoTIFF."""
    with rasterio.open(path) as src:
        raw = src.tags().get("clouds_decoded", "{}")
    return json.loads(raw).get("provenance", {})


def get_band_names(path: Path) -> list[str]:
    """Derive band names from a GeoTIFF's embedded metadata."""
    prov = read_provenance(path)
    cfg = prov.get("step_config", {})
    step = prov.get("step_name", "")

    if step == "cloud_properties":
        names = list(cfg.get("output_feature_names", []))
        if cfg.get("return_uncertainty", False):
            names.append("uncertainty")
        return names

    if step == "albedo":
        return list(cfg.get("default_albedo", {}).keys())

    if step == "cloud_mask":
        return ["cloud_mask"]

    if step == "cloud_height":
        return ["cloud_height_m"]

    # Fallback: numbered bands
    with rasterio.open(path) as src:
        return [f"band_{i}" for i in range(1, src.count + 1)]


def read_band_by_name(path: Path, name: str) -> np.ndarray:
    """Read a single band from a GeoTIFF by its semantic name."""
    band_names = get_band_names(path)
    idx = band_names.index(name) + 1  # rasterio is 1-indexed
    with rasterio.open(path) as src:
        return src.read(idx)

In [ ]:
# Demonstrate on one scene
example_scene = sorted(OUTPUT_DIR.iterdir())[0]
print(f"Scene: {example_scene.name}\n")

for tif in sorted(example_scene.glob("*.tif")):
    prov = read_provenance(tif)
    bands = get_band_names(tif)
    with rasterio.open(tif) as src:
        shape, dtype = src.shape, src.dtypes[0]
    print(f"{tif.name}")
    print(f"  step:  {prov.get('step_name')}")
    print(f"  shape: {shape}, dtype: {dtype}")
    print(f"  bands: {bands}\n")

## 3. Monthly cloud fraction by thermodynamic phase

Using the per-scene rasters, we classify cloudy pixels as **ice**
(ice/liquid ratio > 0.5) or **liquid** (≤ 0.5) and compute the fraction of total
scene pixels in each category, averaged by month over all years.

We use `read_band_by_name()` to look up `ice_liq_ratio` and `cloud_height_m`
dynamically from each file's metadata.

In [ ]:
def compute_monthly_phase_fractions(tile: str):
    """Return (months, ice_frac, liq_frac) arrays for a tile.

    Uses the DuckDB scene_metadata table to resolve months, and the GeoTIFF
    provenance metadata to resolve band indices.
    """
    ice_counts = {m: 0 for m in range(1, 13)}
    liq_counts = {m: 0 for m in range(1, 13)}
    total_pixels = {m: 0 for m in range(1, 13)}

    # Get scene_id -> month mapping from the database
    scene_months = dict(db.sql(f"""
        SELECT r.scene_id,
               EXTRACT(MONTH FROM sm.sensing_time::TIMESTAMP)::INTEGER AS month
        FROM runs r
        JOIN scene_metadata sm ON r.scene_id = sm.scene_id
        WHERE sm.tile_id = '{tile}' AND r.status = 'done'
    """).fetchall())

    scenes = sorted(d for d in OUTPUT_DIR.iterdir() if tile in d.name)
    for scene_dir in tqdm(scenes, desc=f"Reading data for tile {tile}..."):
        height_path = scene_dir / "cloud_height.tif"
        props_path = scene_dir / "properties.tif"
        if not height_path.exists() or not props_path.exists():
            continue

        # Look up month from database via scene_id
        scene_id = scene_dir.name
        month = scene_months.get(scene_id)
        if month is None:
            continue

        height = read_band_by_name(height_path, "cloud_height_m")
        ice_liq = read_band_by_name(props_path, "ice_liq_ratio")

        valid = np.isfinite(height) & np.isfinite(ice_liq) & (height > 0)
        ilr = ice_liq[valid]

        ice_counts[month] += (ilr > 0.5).sum()
        liq_counts[month] += (ilr <= 0.5).sum()
        total_pixels[month] += valid.size

    months = np.arange(1, 13)
    ice_frac = np.array([ice_counts[m] / max(total_pixels[m], 1) for m in months])
    liq_frac = np.array([liq_counts[m] / max(total_pixels[m], 1) for m in months])
    return months, ice_frac, liq_frac

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

for ax, (tile, label) in zip(axes.flat, TILE_LABELS.items()):
    months, ice_frac, liq_frac = compute_monthly_phase_fractions(tile)
    ax.plot(months, liq_frac * 100, "o-", color="tab:red",
            label="Liquid", linewidth=2, markersize=5)
    ax.plot(months, ice_frac * 100, "o-", color="tab:blue",
            label="Ice", linewidth=2, markersize=5)
    ax.set_title(f"{tile} ({label})")
    ax.set_xticks(months)
    ax.set_xticklabels(MONTH_NAMES, fontsize=9)
    ax.set_ylabel("Cloud fraction (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.5, 12.5)

fig.suptitle("Monthly cloud fraction by thermodynamic phase", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 4. Cloud height distributions by phase

For each month we build a histogram of cloud-top heights, split into ice and
liquid. The result is displayed as mirrored violins: liquid (red) on the left,
ice (blue) on the right. Widths share a single global maximum across all months
so that ice and liquid are directly comparable.

In [ ]:
H_MIN, H_MAX, N_BINS = 0, 15_000, 150
BIN_EDGES = np.linspace(H_MIN, H_MAX, N_BINS + 1)
BIN_CENTRES = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
SIGMA = 2.5


def collect_height_histograms(tile: str):
    """Return (ice_hists, liq_hists) dicts keyed by month (1-12)."""
    ice_hists = {m: np.zeros(N_BINS) for m in range(1, 13)}
    liq_hists = {m: np.zeros(N_BINS) for m in range(1, 13)}

    scene_months = dict(db.sql(f"""
        SELECT r.scene_id,
               EXTRACT(MONTH FROM sm.sensing_time::TIMESTAMP)::INTEGER AS month
        FROM runs r
        JOIN scene_metadata sm ON r.scene_id = sm.scene_id
        WHERE sm.tile_id = '{tile}' AND r.status = 'done'
    """).fetchall())

    scenes = sorted(d for d in OUTPUT_DIR.iterdir() if tile in d.name)
    for scene_dir in tqdm(scenes, desc=f"Collecting height histograms for tile {tile}..."):
        height_path = scene_dir / "cloud_height.tif"
        props_path = scene_dir / "properties.tif"
        if not height_path.exists() or not props_path.exists():
            continue

        month = scene_months.get(scene_dir.name)
        if month is None:
            continue

        height = read_band_by_name(height_path, "cloud_height_m")
        ice_liq = read_band_by_name(props_path, "ice_liq_ratio")

        valid = np.isfinite(height) & np.isfinite(ice_liq) & (height > 0)
        h = height[valid]
        ilr = ice_liq[valid]

        ice_mask = ilr > 0.5
        if ice_mask.any():
            ice_hists[month] += np.histogram(h[ice_mask], bins=BIN_EDGES)[0]
        if (~ice_mask).any():
            liq_hists[month] += np.histogram(h[~ice_mask], bins=BIN_EDGES)[0]

    for m in range(1, 13):
        total = ice_hists[m].sum() + liq_hists[m].sum()
        if total > 0:
            ice_hists[m] = gaussian_filter1d(ice_hists[m] / total, SIGMA)
            liq_hists[m] = gaussian_filter1d(liq_hists[m] / total, SIGMA)

    return ice_hists, liq_hists


def plot_violins(ax, ice_hists, liq_hists, title: str):
    """Draw mirrored violin plot on the given axes."""
    max_density = max(
        max(ice_hists[m].max() for m in range(1, 13)),
        max(liq_hists[m].max() for m in range(1, 13)),
    )
    spacing, hw = 1.0, 0.45

    for i, m in enumerate(range(1, 13)):
        xc = i * spacing
        liq_s = liq_hists[m] / max_density * hw
        ice_s = ice_hists[m] / max_density * hw

        ax.fill_betweenx(BIN_CENTRES / 1000, xc - liq_s, xc,
                         color="tab:red", alpha=0.7, linewidth=0)
        ax.plot(xc - liq_s, BIN_CENTRES / 1000, color="darkred", linewidth=0.6)

        ax.fill_betweenx(BIN_CENTRES / 1000, xc, xc + ice_s,
                         color="tab:blue", alpha=0.7, linewidth=0)
        ax.plot(xc + ice_s, BIN_CENTRES / 1000, color="darkblue", linewidth=0.6)

        ax.axvline(xc, color="grey", linewidth=0.3, zorder=0)

    ax.set_xticks([i * spacing for i in range(12)])
    ax.set_xticklabels(MONTH_NAMES, fontsize=8)
    ax.set_ylabel("Cloud height (km)")
    ax.set_ylim(0, 15)
    ax.set_xlim(-0.6, 11.6)
    ax.set_title(title)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (tile, label) in zip(axes.flat, TILE_LABELS.items()):
    ice_h, liq_h = collect_height_histograms(tile)
    plot_violins(ax, ice_h, liq_h, f"{tile} ({label})")

axes[0, 0].legend(
    handles=[Patch(color="tab:red", alpha=0.7, label="Liquid (left)"),
             Patch(color="tab:blue", alpha=0.7, label="Ice (right)")],
    loc="upper right", fontsize=9,
)

fig.suptitle(
    "Monthly cloud-top height distribution by thermodynamic phase\n"
    "(averaged over 2016\u20132025)",
    fontsize=14, y=1.01,
)
fig.tight_layout()
plt.show()

## Notes

- **Band identification** uses the `clouds_decoded` GeoTIFF tag rather than
  hardcoded indices. The `output_feature_names` field in the step config lists
  the property bands in order; for albedo, band names come from `default_albedo`
  keys.
- **Statistics** come from the project's DuckDB database, which stores per-step
  summary statistics (percentiles, means, class fractions) computed at processing
  time.
- **Ice/liquid classification** uses the `ice_liq_ratio` output from the neural
  inversion model. Values > 0.5 are classified as ice-phase; \u2264 0.5 as liquid.
- **Cloud heights** are derived from stereo parallax detection between Sentinel-2
  bands acquired with sub-second time delays from the push-broom sensor.
- Monthly statistics are averaged across all years (up to 10 per month).
- Violin widths within each plot share a single global maximum, so ice and liquid
  contributions are visually comparable across months.